# Curriculum Learning: Training from Easy to Hard

## Introduction

**Curriculum learning** is a training strategy inspired by how humans and animals learn — starting with simple concepts and gradually progressing to more complex ones. Rather than throwing all training examples at a model randomly, we carefully order them from easy to hard.

### What We'll Learn

In this notebook, we'll explore:

1. **Why order matters** — How training sequence affects learning
2. **Difficulty metrics** — Different ways to measure example "hardness"
3. **Curriculum strategies** — Baby steps, spiral curricula, and self-paced learning
4. **Practical implementations** — Image classification and sequence learning tasks
5. **When curricula help** — Noisy data, hard optimization, domain shift
6. **Anti-curriculum** — The controversial idea of starting with hard examples

### Key Intuition

Just as a math teacher wouldn't start with calculus before teaching arithmetic, neural networks can benefit from a structured progression of training examples. Starting with easier examples can:
- Help the model find better optimization paths
- Avoid bad local minima early in training
- Improve generalization on the hardest examples
- Speed up convergence

But defining "easy" and "hard" is non-trivial — we'll explore multiple approaches.

## Setup

In [ ]:

    get_device,
    set_seed,
    CIFAR10_MEAN,
    CIFAR10_STD,
    CIFAR10_CLASSES,
)

%load_ext autoreload
%autoreload 2

### Configure reproducibility and device

In [ ]:
from aiml_notebooks import (

In [ ]:
set_seed(42)
device = get_device()
print(f"Using device: {device}")

## 1. Curriculum Learning Fundamentals

### What is "Difficulty"?

Before we can order examples from easy to hard, we need to define what makes an example "difficult". There are several perspectives:

1. **Loss-based**: Examples with higher loss are harder (model-dependent)
2. **Confidence-based**: Examples with lower prediction confidence are harder (model-dependent)
3. **Domain knowledge**: Manual features like length, complexity, noise (model-independent)
4. **Model disagreement**: Examples where different models disagree (ensemble-based)

Let's implement these different difficulty scoring functions.

### Domain Knowledge Difficulty: Image Complexity

For images, we can use hand-crafted features to estimate difficulty without needing a model. Simple metrics include:
- **Edge density**: More edges = more complex
- **Variance**: Higher variance = more information
- **Entropy**: Higher entropy = more randomness

Let's start with a simple variance-based difficulty metric.

In [ ]:
def compute_image_variance(image: torch.Tensor) -> float:
    """
    Compute variance of image pixels as a proxy for visual complexity.
    Higher variance = more detail/texture = harder.
    
    Args:
        image: Tensor of shape (C, H, W)
    Returns:
        Variance score (scalar)
    """
    return float(image.var())

def compute_edge_density(image: torch.Tensor) -> float:
    """
    Compute edge density using Sobel filter.
    More edges = more complex structure = harder.
    
    Args:
        image: Tensor of shape (C, H, W)
    Returns:
        Edge density score (scalar)
    """
    # Convert to grayscale if needed
    if image.shape[0] == 3:
        grayscale = 0.299 * image[0] + 0.587 * image[1] + 0.114 * image[2]
    else:
        grayscale = image[0]
    
    # Simple gradient magnitude
    dx = torch.abs(grayscale[:, 1:] - grayscale[:, :-1])
    dy = torch.abs(grayscale[1:, :] - grayscale[:-1, :])
    
    edge_density = (dx.sum() + dy.sum()) / grayscale.numel()
    return float(edge_density)

# Test on a simple example
simple_image = torch.ones(3, 32, 32) * 0.5  # uniform gray
complex_image = torch.randn(3, 32, 32)  # random noise

print(f"Simple image variance: {compute_image_variance(simple_image):.4f}")
print(f"Complex image variance: {compute_image_variance(complex_image):.4f}")
print(f"Simple image edge density: {compute_edge_density(simple_image):.4f}")
print(f"Complex image edge density: {compute_edge_density(complex_image):.4f}")

### Model-Based Difficulty: Loss and Confidence

Once we have a trained model, we can use its predictions to estimate difficulty:
- **Loss**: Higher loss = model struggles = harder example
- **Confidence**: Lower max probability = model uncertain = harder example

These are model-dependent metrics that adapt as the model learns.

In [ ]:
def compute_loss_difficulty(
    model: nn.Module,
    images: torch.Tensor,
    labels: torch.Tensor,
    device: torch.device
) -> np.ndarray:
    """
    Compute per-example loss as difficulty score.
    
    Args:
        model: Neural network
        images: Batch of images (N, C, H, W)
        labels: Batch of labels (N,)
        device: Device to run on
    Returns:
        Array of loss values (N,)
    """
    model.eval()
    with torch.no_grad():
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        losses = F.cross_entropy(logits, labels, reduction='none')
    return losses.cpu().numpy()

def compute_confidence_difficulty(
    model: nn.Module,
    images: torch.Tensor,
    device: torch.device
) -> np.ndarray:
    """
    Compute 1 - max_confidence as difficulty score.
    Low confidence = high difficulty.
    
    Args:
        model: Neural network
        images: Batch of images (N, C, H, W)
        device: Device to run on
    Returns:
        Array of difficulty scores (N,)
    """
    model.eval()
    with torch.no_grad():
        images = images.to(device)
        logits = model(images)
        probs = F.softmax(logits, dim=1)
        max_probs = probs.max(dim=1).values
        difficulty = 1.0 - max_probs
    return difficulty.cpu().numpy()

print("Difficulty scoring functions ready!")

## 2. Curriculum Strategies

There are several ways to organize a curriculum:

1. **Baby Steps**: Start with easiest examples, gradually introduce harder ones
2. **One-Pass**: Single progression from easy to hard (no repetition)
3. **Spiral**: Revisit examples at increasing difficulty levels
4. **Self-Paced**: Model decides its own learning pace

Let's implement a curriculum scheduler that controls which examples to show at each training step.

In [ ]:
from dataclasses import dataclass

In [ ]:
@dataclass
class CurriculumConfig:
    """Configuration for curriculum learning."""
    strategy: str  # 'baby_steps', 'one_pass', 'spiral', 'self_paced'
    initial_ratio: float = 0.2  # Start with easiest 20% of data
    final_ratio: float = 1.0  # Eventually use all data
    growth_rate: float = 0.1  # How fast to add harder examples
    warmup_epochs: int = 5  # Epochs before starting to add harder examples
    
class CurriculumScheduler:
    """
    Manages which training examples to use at each epoch.
    """
    def __init__(self, difficulty_scores: np.ndarray, config: CurriculumConfig):
        """
        Args:
            difficulty_scores: Array of difficulty scores for each example
            config: Curriculum configuration
        """
        self.difficulty_scores = difficulty_scores
        self.config = config
        # Sort indices by difficulty (easy to hard)
        self.sorted_indices = np.argsort(difficulty_scores)
        self.n_examples = len(difficulty_scores)
        
    def get_indices(self, epoch: int) -> np.ndarray:
        """
        Get indices of examples to use at this epoch.
        
        Args:
            epoch: Current epoch number (0-indexed)
        Returns:
            Array of indices to use
        """
        if epoch < self.config.warmup_epochs:
            # Use only easiest examples during warmup
            ratio = self.config.initial_ratio
        else:
            # Gradually increase ratio
            epochs_since_warmup = epoch - self.config.warmup_epochs
            ratio = min(
                self.config.final_ratio,
                self.config.initial_ratio + epochs_since_warmup * self.config.growth_rate
            )
        
        n_examples = int(self.n_examples * ratio)
        
        if self.config.strategy == 'baby_steps':
            # Take easiest n_examples, shuffle them
            indices = self.sorted_indices[:n_examples]
            np.random.shuffle(indices)
            return indices
        elif self.config.strategy == 'one_pass':
            # Take easiest n_examples in order
            return self.sorted_indices[:n_examples]
        else:
            raise ValueError(f"Unknown strategy: {self.config.strategy}")

# Example usage
example_difficulties = np.random.rand(1000)
config = CurriculumConfig(strategy='baby_steps', initial_ratio=0.2, growth_rate=0.15)
scheduler = CurriculumScheduler(example_difficulties, config)

print(f"Epoch 0 (warmup): {len(scheduler.get_indices(0))} examples")
print(f"Epoch 5 (post-warmup): {len(scheduler.get_indices(5))} examples")
print(f"Epoch 10: {len(scheduler.get_indices(10))} examples")

### Visualizing Curriculum Progression

Let's visualize how the curriculum grows over epochs — showing which percentile of difficulty we're including.

In [ ]:
from torch.utils.data import Dataset, DataLoader, Subset

In [ ]:
def plot_curriculum_progression(scheduler: CurriculumScheduler, max_epochs: int = 20):
    """
    Plot how many examples and their difficulty range changes over epochs.
    """
    epochs = range(max_epochs)
    n_examples = []
    max_difficulty = []
    mean_difficulty = []
    
    for epoch in epochs:
        indices = scheduler.get_indices(epoch)
        n_examples.append(len(indices))
        max_difficulty.append(scheduler.difficulty_scores[indices].max())
        mean_difficulty.append(scheduler.difficulty_scores[indices].mean())
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Plot number of examples
    axes[0].plot(epochs, n_examples, marker='o')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Number of Examples')
    axes[0].set_title('Dataset Size Over Time')
    axes[0].grid(True, alpha=0.3)
    
    # Plot difficulty range
    axes[1].plot(epochs, mean_difficulty, marker='o', label='Mean Difficulty')
    axes[1].plot(epochs, max_difficulty, marker='s', label='Max Difficulty')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Difficulty Score')
    axes[1].set_title('Difficulty Progression')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_curriculum_progression(scheduler, max_epochs=20)

Notice how both the dataset size and the maximum difficulty increase gradually. This is the **baby steps** strategy — we start with a small set of easy examples and progressively add harder ones.

## 3. Implementation: Image Classification with CIFAR-10

Let's apply curriculum learning to CIFAR-10 image classification. We'll:
1. Load CIFAR-10
2. Compute difficulty scores using image variance
3. Train two models: one with curriculum, one with random sampling
4. Compare their learning curves

### Load and prepare CIFAR-10 dataset

In [ ]:
from torchvision import datasets, transforms

In [ ]:
# Define transforms
transform_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

# Load datasets
train_dataset = datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform_train
)

test_dataset = datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform_test
)

# Use subset for faster experimentation
n_train = 10000  # Use 10k examples instead of 50k for speed
train_indices = np.random.choice(len(train_dataset), n_train, replace=False)
train_dataset = Subset(train_dataset, train_indices)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

### Compute difficulty scores for all training images

We'll use **edge density** as our difficulty metric — images with more edges are considered harder because they have more complex structure.

In [ ]:
from typing import List, Tuple, Callable, Optionalfrom tqdm.auto import tqdm

In [ ]:
def compute_dataset_difficulties(
    dataset: Dataset,
    difficulty_fn: Callable,
    denormalize: bool = True
) -> np.ndarray:
    """
    Compute difficulty scores for all examples in a dataset.
    
    Args:
        dataset: PyTorch dataset
        difficulty_fn: Function that takes an image and returns a scalar score
        denormalize: Whether to denormalize images first
    Returns:
        Array of difficulty scores
    """
    difficulties = []
    
    for i in tqdm(range(len(dataset)), desc="Computing difficulties"):
        image, _ = dataset[i]
        
        # Denormalize if needed for visual features
        if denormalize:
            mean = torch.tensor(CIFAR10_MEAN).view(3, 1, 1)
            std = torch.tensor(CIFAR10_STD).view(3, 1, 1)
            image = image * std + mean
        
        difficulty = difficulty_fn(image)
        difficulties.append(difficulty)
    
    return np.array(difficulties)

# Compute edge-based difficulties
train_difficulties = compute_dataset_difficulties(
    train_dataset,
    compute_edge_density,
    denormalize=True
)

print(f"Difficulty range: [{train_difficulties.min():.4f}, {train_difficulties.max():.4f}]")
print(f"Mean difficulty: {train_difficulties.mean():.4f}")

### Visualize easy vs hard examples

Let's look at what the model considers "easy" versus "hard" based on edge density.

In [ ]:
def show_examples_by_difficulty(
    dataset: Dataset,
    difficulties: np.ndarray,
    n_examples: int = 5
):
    """
    Show easiest and hardest examples side by side.
    """
    sorted_indices = np.argsort(difficulties)
    easiest_indices = sorted_indices[:n_examples]
    hardest_indices = sorted_indices[-n_examples:]
    
    fig, axes = plt.subplots(2, n_examples, figsize=(12, 5))
    
    mean = torch.tensor(CIFAR10_MEAN).view(3, 1, 1)
    std = torch.tensor(CIFAR10_STD).view(3, 1, 1)
    
    # Show easiest examples
    for i, idx in enumerate(easiest_indices):
        image, label = dataset[idx]
        image = image * std + mean
        image = torch.clamp(image, 0, 1)
        axes[0, i].imshow(image.permute(1, 2, 0))
        axes[0, i].set_title(f"Easy\n{CIFAR10_CLASSES[label]}\n{difficulties[idx]:.3f}")
        axes[0, i].axis('off')
    
    # Show hardest examples
    for i, idx in enumerate(hardest_indices):
        image, label = dataset[idx]
        image = image * std + mean
        image = torch.clamp(image, 0, 1)
        axes[1, i].imshow(image.permute(1, 2, 0))
        axes[1, i].set_title(f"Hard\n{CIFAR10_CLASSES[label]}\n{difficulties[idx]:.3f}")
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.show()

show_examples_by_difficulty(train_dataset, train_difficulties, n_examples=5)

Notice that **easy** examples tend to have simpler shapes and uniform colors (low edge density), while **hard** examples have more texture and detail (high edge density).

### Define a simple CNN classifier

In [ ]:
class SimpleCNN(nn.Module):
    """Simple CNN for CIFAR-10 classification."""
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x):
        # Input: (N, 3, 32, 32)
        x = self.pool(F.relu(self.conv1(x)))  # (N, 32, 16, 16)
        x = self.pool(F.relu(self.conv2(x)))  # (N, 64, 8, 8)
        x = self.pool(F.relu(self.conv3(x)))  # (N, 128, 4, 4)
        x = x.view(x.size(0), -1)  # (N, 128*4*4)
        x = self.dropout(F.relu(self.fc1(x)))  # (N, 256)
        x = self.fc2(x)  # (N, 10)
        return x

model = SimpleCNN()
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

### Training function with curriculum support

In [ ]:
def train_with_curriculum(
    model: nn.Module,
    dataset: Dataset,
    test_loader: DataLoader,
    scheduler: Optional[CurriculumScheduler],
    epochs: int,
    batch_size: int,
    lr: float,
    device: torch.device
) -> dict:
    """
    Train with optional curriculum learning.
    
    Args:
        model: Neural network to train
        dataset: Full training dataset
        test_loader: Test DataLoader for evaluation
        scheduler: Curriculum scheduler (None for random training)
        epochs: Number of training epochs
        batch_size: Batch size
        lr: Learning rate
        device: Device to train on
    Returns:
        Dictionary with training history
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    history = {
        'train_loss': [],
        'train_acc': [],
        'test_acc': [],
        'n_examples': []
    }
    
    for epoch in range(epochs):
        # Get indices for this epoch
        if scheduler is not None:
            indices = scheduler.get_indices(epoch)
        else:
            indices = np.arange(len(dataset))
        
        # Create subset and loader
        epoch_dataset = Subset(dataset, indices)
        train_loader = DataLoader(
            epoch_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=2
        )
        
        # Training
        model.train()
        total_loss = 0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = F.cross_entropy(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        train_loss = total_loss / total
        train_acc = 100.0 * correct / total
        
        # Evaluation
        model.eval()
        test_correct = 0
        test_total = 0
        
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = outputs.max(1)
                test_total += labels.size(0)
                test_correct += predicted.eq(labels).sum().item()
        
        test_acc = 100.0 * test_correct / test_total
        
        # Record history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        history['n_examples'].append(len(indices))
        
        print(f"Epoch {epoch+1}/{epochs} - "
              f"Examples: {len(indices):5d} - "
              f"Loss: {train_loss:.4f} - "
              f"Train Acc: {train_acc:.2f}% - "
              f"Test Acc: {test_acc:.2f}%")
    
    return history

print("Training function ready!")

### Train baseline model (random sampling, no curriculum)

In [ ]:
# Create test loader
test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

# Train baseline
set_seed(42)
baseline_model = SimpleCNN()
print("Training baseline (random sampling)...")
baseline_history = train_with_curriculum(
    model=baseline_model,
    dataset=train_dataset,
    test_loader=test_loader,
    scheduler=None,  # No curriculum
    epochs=15,
    batch_size=128,
    lr=0.001,
    device=device
)

### Train model with curriculum learning

In [ ]:
# Create curriculum scheduler
curriculum_config = CurriculumConfig(
    strategy='baby_steps',
    initial_ratio=0.3,  # Start with easiest 30%
    final_ratio=1.0,
    growth_rate=0.1,  # Add 10% more each epoch
    warmup_epochs=3
)

curriculum_scheduler = CurriculumScheduler(train_difficulties, curriculum_config)

# Train with curriculum
set_seed(42)
curriculum_model = SimpleCNN()
print("\nTraining with curriculum learning...")
curriculum_history = train_with_curriculum(
    model=curriculum_model,
    dataset=train_dataset,
    test_loader=test_loader,
    scheduler=curriculum_scheduler,
    epochs=15,
    batch_size=128,
    lr=0.001,
    device=device
)

### Compare curriculum vs baseline learning curves

Now let's visualize the difference between curriculum learning and random sampling.

In [ ]:
def plot_comparison(baseline_history: dict, curriculum_history: dict):
    """
    Compare baseline vs curriculum learning.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    epochs = range(1, len(baseline_history['train_loss']) + 1)
    
    # Training loss
    axes[0].plot(epochs, baseline_history['train_loss'], marker='o', label='Baseline')
    axes[0].plot(epochs, curriculum_history['train_loss'], marker='s', label='Curriculum')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Training Loss')
    axes[0].set_title('Training Loss Comparison')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Training accuracy
    axes[1].plot(epochs, baseline_history['train_acc'], marker='o', label='Baseline')
    axes[1].plot(epochs, curriculum_history['train_acc'], marker='s', label='Curriculum')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Training Accuracy (%)')
    axes[1].set_title('Training Accuracy Comparison')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Test accuracy
    axes[2].plot(epochs, baseline_history['test_acc'], marker='o', label='Baseline')
    axes[2].plot(epochs, curriculum_history['test_acc'], marker='s', label='Curriculum')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Test Accuracy (%)')
    axes[2].set_title('Test Accuracy Comparison')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print final results
    print("\nFinal Results:")
    print(f"Baseline - Test Acc: {baseline_history['test_acc'][-1]:.2f}%")
    print(f"Curriculum - Test Acc: {curriculum_history['test_acc'][-1]:.2f}%")
    print(f"Improvement: {curriculum_history['test_acc'][-1] - baseline_history['test_acc'][-1]:.2f}%")

plot_comparison(baseline_history, curriculum_history)

**Key Observations**:

The curriculum model often shows:
- **Faster initial learning** — starting with easy examples helps the model quickly learn basic features
- **Smoother convergence** — gradual difficulty increase avoids sudden jumps in loss
- **Better final performance** — the model may generalize better to test data

However, results can vary depending on the task, difficulty metric, and curriculum schedule.

## 4. Self-Paced Learning

In **self-paced learning**, the model chooses its own curriculum based on which examples it finds easy. Instead of a fixed schedule, we dynamically select examples with lower loss.

The algorithm has an **age parameter** that controls the pace:
- Start with only the easiest examples
- Gradually increase threshold to include harder examples
- Model adapts the pace to its own learning progress

### Self-Paced Learning Implementation

In [ ]:
class SelfPacedLearner:
    """
    Self-paced learning: model chooses its own curriculum.
    """
    def __init__(self, initial_threshold: float = 1.0, growth_rate: float = 0.1):
        """
        Args:
            initial_threshold: Initial loss threshold (lower = stricter)
            growth_rate: How fast to increase threshold per epoch
        """
        self.threshold = initial_threshold
        self.growth_rate = growth_rate
    
    def select_examples(
        self,
        model: nn.Module,
        dataset: Dataset,
        device: torch.device,
        batch_size: int = 256
    ) -> np.ndarray:
        """
        Select examples based on current model loss.
        
        Returns:
            Indices of selected examples
        """
        model.eval()
        all_losses = []
        
        # Compute loss for all examples
        temp_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
        
        with torch.no_grad():
            for images, labels in temp_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                losses = F.cross_entropy(outputs, labels, reduction='none')
                all_losses.append(losses.cpu().numpy())
        
        all_losses = np.concatenate(all_losses)
        
        # Select examples below threshold
        selected_indices = np.where(all_losses <= self.threshold)[0]
        
        # Ensure we have at least some examples
        if len(selected_indices) < 100:
            # Fall back to selecting easiest N examples
            n_examples = max(100, int(0.2 * len(dataset)))
            selected_indices = np.argsort(all_losses)[:n_examples]
        
        return selected_indices
    
    def step(self):
        """Increase threshold for next epoch."""
        self.threshold += self.growth_rate

print("Self-paced learner ready!")

### Training with self-paced learning

In [ ]:
def train_self_paced(
    model: nn.Module,
    dataset: Dataset,
    test_loader: DataLoader,
    learner: SelfPacedLearner,
    epochs: int,
    batch_size: int,
    lr: float,
    device: torch.device
) -> dict:
    """
    Train with self-paced learning.
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    history = {
        'train_loss': [],
        'train_acc': [],
        'test_acc': [],
        'n_examples': [],
        'threshold': []
    }
    
    for epoch in range(epochs):
        # Select examples based on current model
        selected_indices = learner.select_examples(model, dataset, device)
        
        # Create loader for selected examples
        epoch_dataset = Subset(dataset, selected_indices)
        train_loader = DataLoader(
            epoch_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=2
        )
        
        # Training
        model.train()
        total_loss = 0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = F.cross_entropy(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        train_loss = total_loss / total
        train_acc = 100.0 * correct / total
        
        # Evaluation
        model.eval()
        test_correct = 0
        test_total = 0
        
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = outputs.max(1)
                test_total += labels.size(0)
                test_correct += predicted.eq(labels).sum().item()
        
        test_acc = 100.0 * test_correct / test_total
        
        # Record history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        history['n_examples'].append(len(selected_indices))
        history['threshold'].append(learner.threshold)
        
        print(f"Epoch {epoch+1}/{epochs} - "
              f"Examples: {len(selected_indices):5d} - "
              f"Threshold: {learner.threshold:.2f} - "
              f"Loss: {train_loss:.4f} - "
              f"Test Acc: {test_acc:.2f}%")
        
        # Update threshold for next epoch
        learner.step()
    
    return history

# Train with self-paced learning
set_seed(42)
selfpaced_model = SimpleCNN()
selfpaced_learner = SelfPacedLearner(initial_threshold=0.8, growth_rate=0.15)

print("Training with self-paced learning...")
selfpaced_history = train_self_paced(
    model=selfpaced_model,
    dataset=train_dataset,
    test_loader=test_loader,
    learner=selfpaced_learner,
    epochs=15,
    batch_size=128,
    lr=0.001,
    device=device
)

### Compare all three approaches

Let's compare baseline, fixed curriculum, and self-paced learning side by side.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs = range(1, len(baseline_history['test_acc']) + 1)

# Test accuracy comparison
axes[0].plot(epochs, baseline_history['test_acc'], marker='o', label='Baseline')
axes[0].plot(epochs, curriculum_history['test_acc'], marker='s', label='Fixed Curriculum')
axes[0].plot(epochs, selfpaced_history['test_acc'], marker='^', label='Self-Paced')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title('Test Accuracy: All Methods')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Dataset size over time
axes[1].plot(epochs, curriculum_history['n_examples'], marker='s', label='Fixed Curriculum')
axes[1].plot(epochs, selfpaced_history['n_examples'], marker='^', label='Self-Paced')
axes[1].axhline(y=len(train_dataset), color='gray', linestyle='--', label='Full Dataset')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Number of Examples')
axes[1].set_title('Active Dataset Size')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nFinal Test Accuracy:")
print(f"Baseline: {baseline_history['test_acc'][-1]:.2f}%")
print(f"Fixed Curriculum: {curriculum_history['test_acc'][-1]:.2f}%")
print(f"Self-Paced: {selfpaced_history['test_acc'][-1]:.2f}%")

**Key Insight**: Self-paced learning adapts the curriculum based on the model's actual performance, potentially leading to more efficient learning than a fixed schedule.

## 5. Sequence Learning with Length Curriculum

Curriculum learning is especially powerful for sequence tasks. We'll demonstrate with a simple **arithmetic task**: learning to add numbers.

The curriculum:
1. **Easy**: 1-digit addition (3 + 5 = 8)
2. **Medium**: 2-digit addition (12 + 34 = 46)
3. **Hard**: 3-digit addition (123 + 456 = 579)

This is a natural curriculum based on sequence length.

### Generate arithmetic dataset with varying difficulty

In [ ]:
class ArithmeticDataset(Dataset):
    """
    Dataset for addition problems: 'A+B=C'
    """
    def __init__(self, num_examples: int, max_digits: int, seed: int = 42):
        np.random.seed(seed)
        self.examples = []
        self.difficulties = []
        
        # Create character vocabulary
        self.chars = '0123456789+='
        self.char_to_idx = {c: i for i, c in enumerate(self.chars)}
        self.idx_to_char = {i: c for c, i in self.char_to_idx.items()}
        
        # Generate examples with different number of digits
        for _ in range(num_examples):
            # Random number of digits (1 to max_digits)
            n_digits = np.random.randint(1, max_digits + 1)
            
            # Generate two random numbers
            max_val = 10 ** n_digits - 1
            min_val = 10 ** (n_digits - 1) if n_digits > 1 else 0
            
            a = np.random.randint(min_val, max_val + 1)
            b = np.random.randint(min_val, max_val + 1)
            c = a + b
            
            # Create equation string
            equation = f"{a}+{b}={c}"
            self.examples.append(equation)
            self.difficulties.append(n_digits)  # Difficulty = number of digits
        
        self.difficulties = np.array(self.difficulties)
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        equation = self.examples[idx]
        # Encode as indices
        indices = [self.char_to_idx[c] for c in equation]
        return torch.tensor(indices, dtype=torch.long), len(equation)
    
    def decode(self, indices):
        """Convert indices back to string."""
        return ''.join([self.idx_to_char[int(i)] for i in indices])

# Create dataset
arithmetic_dataset = ArithmeticDataset(num_examples=5000, max_digits=3)

print(f"Dataset size: {len(arithmetic_dataset)}")
print(f"Vocabulary size: {len(arithmetic_dataset.chars)}")
print(f"\nExample problems:")
for i in range(5):
    print(f"  {arithmetic_dataset.examples[i]} (difficulty: {arithmetic_dataset.difficulties[i]})")

### Visualize difficulty distribution

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(arithmetic_dataset.difficulties, bins=np.arange(0.5, 4.5, 1), edgecolor='black')
plt.xlabel('Number of Digits')
plt.ylabel('Count')
plt.title('Distribution of Problem Difficulty')
plt.xticks([1, 2, 3])
plt.grid(True, alpha=0.3, axis='y')
plt.show()

### Simple sequence-to-sequence model for arithmetic

We'll use a basic LSTM to learn the addition pattern.

In [ ]:
class ArithmeticLSTM(nn.Module):
    """LSTM for learning arithmetic."""
    def __init__(self, vocab_size: int, embed_dim: int = 32, hidden_dim: int = 64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x):
        # x: (batch_size, seq_len)
        embedded = self.embedding(x)  # (batch_size, seq_len, embed_dim)
        output, _ = self.lstm(embedded)  # (batch_size, seq_len, hidden_dim)
        logits = self.fc(output)  # (batch_size, seq_len, vocab_size)
        return logits

model = ArithmeticLSTM(vocab_size=len(arithmetic_dataset.chars))
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

### Collate function for variable-length sequences

In [ ]:
def arithmetic_collate_fn(batch):
    """
    Pad sequences to same length in batch.
    """
    sequences, lengths = zip(*batch)
    max_len = max(lengths)
    
    # Pad with zeros
    padded = torch.zeros(len(sequences), max_len, dtype=torch.long)
    for i, seq in enumerate(sequences):
        padded[i, :len(seq)] = seq
    
    return padded, torch.tensor(lengths)

print("Collate function ready!")

### Training function for sequence model

In [ ]:
def train_arithmetic(
    model: nn.Module,
    dataset: ArithmeticDataset,
    scheduler: Optional[CurriculumScheduler],
    epochs: int,
    batch_size: int,
    lr: float,
    device: torch.device
) -> dict:
    """
    Train arithmetic model with optional curriculum.
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    history = {'loss': [], 'accuracy': [], 'n_examples': []}
    
    for epoch in range(epochs):
        # Get indices for this epoch
        if scheduler is not None:
            indices = scheduler.get_indices(epoch)
        else:
            indices = np.arange(len(dataset))
        
        epoch_dataset = Subset(dataset, indices)
        loader = DataLoader(
            epoch_dataset,
            batch_size=batch_size,
            shuffle=True,
            collate_fn=arithmetic_collate_fn
        )
        
        model.train()
        total_loss = 0
        correct_chars = 0
        total_chars = 0
        
        for sequences, lengths in loader:
            sequences = sequences.to(device)
            
            # Input: all but last token, Target: all but first token
            input_seq = sequences[:, :-1]
            target_seq = sequences[:, 1:]
            
            optimizer.zero_grad()
            logits = model(input_seq)
            
            # Flatten for loss computation
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                target_seq.reshape(-1),
                ignore_index=0  # Ignore padding
            )
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
            # Compute accuracy
            predictions = logits.argmax(dim=-1)
            mask = target_seq != 0
            correct_chars += (predictions == target_seq)[mask].sum().item()
            total_chars += mask.sum().item()
        
        avg_loss = total_loss / len(loader)
        accuracy = 100.0 * correct_chars / total_chars
        
        history['loss'].append(avg_loss)
        history['accuracy'].append(accuracy)
        history['n_examples'].append(len(indices))
        
        print(f"Epoch {epoch+1}/{epochs} - "
              f"Examples: {len(indices):4d} - "
              f"Loss: {avg_loss:.4f} - "
              f"Char Acc: {accuracy:.2f}%")
    
    return history

print("Training function ready!")

### Train baseline (no curriculum)

In [ ]:
set_seed(42)
arithmetic_baseline = ArithmeticLSTM(vocab_size=len(arithmetic_dataset.chars))

print("Training baseline (random)...")
arithmetic_baseline_history = train_arithmetic(
    model=arithmetic_baseline,
    dataset=arithmetic_dataset,
    scheduler=None,
    epochs=20,
    batch_size=64,
    lr=0.001,
    device=device
)

### Train with length-based curriculum

In [ ]:
# Create curriculum based on difficulty (number of digits)
arithmetic_curriculum_config = CurriculumConfig(
    strategy='baby_steps',
    initial_ratio=0.3,
    growth_rate=0.1,
    warmup_epochs=2
)

arithmetic_scheduler = CurriculumScheduler(
    arithmetic_dataset.difficulties,
    arithmetic_curriculum_config
)

set_seed(42)
arithmetic_curriculum = ArithmeticLSTM(vocab_size=len(arithmetic_dataset.chars))

print("\nTraining with curriculum (easy to hard)...")
arithmetic_curriculum_history = train_arithmetic(
    model=arithmetic_curriculum,
    dataset=arithmetic_dataset,
    scheduler=arithmetic_scheduler,
    epochs=20,
    batch_size=64,
    lr=0.001,
    device=device
)

### Compare sequence learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs = range(1, len(arithmetic_baseline_history['loss']) + 1)

# Loss comparison
axes[0].plot(epochs, arithmetic_baseline_history['loss'], marker='o', label='Baseline')
axes[0].plot(epochs, arithmetic_curriculum_history['loss'], marker='s', label='Curriculum')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss: Arithmetic Task')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy comparison
axes[1].plot(epochs, arithmetic_baseline_history['accuracy'], marker='o', label='Baseline')
axes[1].plot(epochs, arithmetic_curriculum_history['accuracy'], marker='s', label='Curriculum')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Character Accuracy (%)')
axes[1].set_title('Character-Level Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nFinal Character Accuracy:")
print(f"Baseline: {arithmetic_baseline_history['accuracy'][-1]:.2f}%")
print(f"Curriculum: {arithmetic_curriculum_history['accuracy'][-1]:.2f}%")

### Test on specific difficulty levels

Let's evaluate both models on problems of different difficulty to see if the curriculum helps with harder examples.

In [ ]:
def evaluate_by_difficulty(
    model: nn.Module,
    dataset: ArithmeticDataset,
    device: torch.device
) -> dict:
    """
    Evaluate model accuracy for each difficulty level.
    """
    model.eval()
    results = {1: {'correct': 0, 'total': 0},
               2: {'correct': 0, 'total': 0},
               3: {'correct': 0, 'total': 0}}
    
    with torch.no_grad():
        for i in range(len(dataset)):
            sequence, _ = dataset[i]
            difficulty = dataset.difficulties[i]
            
            # Predict
            input_seq = sequence[:-1].unsqueeze(0).to(device)
            logits = model(input_seq)
            predictions = logits.argmax(dim=-1).squeeze(0)
            
            # Check if entire sequence is correct
            target = sequence[1:]
            correct = (predictions.cpu() == target).all().item()
            
            results[difficulty]['total'] += 1
            results[difficulty]['correct'] += correct
    
    # Compute accuracies
    accuracies = {}
    for diff in [1, 2, 3]:
        if results[diff]['total'] > 0:
            accuracies[diff] = 100.0 * results[diff]['correct'] / results[diff]['total']
        else:
            accuracies[diff] = 0.0
    
    return accuracies

baseline_acc = evaluate_by_difficulty(arithmetic_baseline, arithmetic_dataset, device)
curriculum_acc = evaluate_by_difficulty(arithmetic_curriculum, arithmetic_dataset, device)

print("Accuracy by Difficulty Level:\n")
print("Difficulty | Baseline | Curriculum")
print("-" * 35)
for diff in [1, 2, 3]:
    print(f"{diff} digit(s) | {baseline_acc[diff]:6.2f}% | {curriculum_acc[diff]:6.2f}%")

**Observation**: The curriculum-trained model often performs better on harder examples (2-3 digits) because it learned the fundamentals on easier examples first.

## 6. Transfer Curriculum

**Transfer curriculum** uses an easier **auxiliary task** as a stepping stone to the main task. The idea:
1. Train on a simpler related task first
2. Fine-tune on the actual target task

Example: Learn to classify shapes (easy) before classifying objects (hard).

For demonstration, we'll use CIFAR-10 and create a simpler "grayscale" version as the auxiliary task.

### Create auxiliary task: grayscale classification

Grayscale images remove color information, making the task simpler.

In [ ]:
class GrayscaleCIFAR10(Dataset):
    """
    Wrapper that converts CIFAR-10 to grayscale.
    """
    def __init__(self, base_dataset):
        self.base_dataset = base_dataset
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        image, label = self.base_dataset[idx]
        # Convert to grayscale by averaging channels
        if image.shape[0] == 3:
            gray = image.mean(dim=0, keepdim=True)
            # Repeat to 3 channels for compatibility with our CNN
            gray = gray.repeat(3, 1, 1)
            return gray, label
        return image, label

# Create grayscale version of training set
gray_train_dataset = GrayscaleCIFAR10(train_dataset)

print("Grayscale dataset created!")

### Visualize color vs grayscale

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))

mean = torch.tensor(CIFAR10_MEAN).view(3, 1, 1)
std = torch.tensor(CIFAR10_STD).view(3, 1, 1)

for i in range(5):
    # Color version
    color_img, label = train_dataset[i]
    color_img = color_img * std + mean
    color_img = torch.clamp(color_img, 0, 1)
    axes[0, i].imshow(color_img.permute(1, 2, 0))
    axes[0, i].set_title(f"Color\n{CIFAR10_CLASSES[label]}")
    axes[0, i].axis('off')
    
    # Grayscale version
    gray_img, _ = gray_train_dataset[i]
    gray_img = gray_img * std + mean
    gray_img = torch.clamp(gray_img, 0, 1)
    axes[1, i].imshow(gray_img.permute(1, 2, 0))
    axes[1, i].set_title(f"Grayscale")
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

### Transfer curriculum training

We'll train on grayscale first, then fine-tune on color.

In [ ]:
def train_transfer_curriculum(
    model: nn.Module,
    auxiliary_dataset: Dataset,
    main_dataset: Dataset,
    test_loader: DataLoader,
    aux_epochs: int,
    main_epochs: int,
    batch_size: int,
    lr: float,
    device: torch.device
) -> dict:
    """
    Train with transfer curriculum: auxiliary task → main task.
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    history = {'phase': [], 'test_acc': []}
    
    # Phase 1: Train on auxiliary task (grayscale)
    print("Phase 1: Training on auxiliary task (grayscale)...")
    aux_loader = DataLoader(auxiliary_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    
    for epoch in range(aux_epochs):
        model.train()
        for images, labels in aux_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = F.cross_entropy(outputs, labels)
            loss.backward()
            optimizer.step()
        
        # Evaluate
        model.eval()
        test_correct = 0
        test_total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = outputs.max(1)
                test_total += labels.size(0)
                test_correct += predicted.eq(labels).sum().item()
        
        test_acc = 100.0 * test_correct / test_total
        history['phase'].append(f'aux_{epoch+1}')
        history['test_acc'].append(test_acc)
        print(f"  Aux Epoch {epoch+1}/{aux_epochs} - Test Acc: {test_acc:.2f}%")
    
    # Phase 2: Fine-tune on main task (color)
    print("\nPhase 2: Fine-tuning on main task (color)...")
    main_loader = DataLoader(main_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    
    for epoch in range(main_epochs):
        model.train()
        for images, labels in main_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = F.cross_entropy(outputs, labels)
            loss.backward()
            optimizer.step()
        
        # Evaluate
        model.eval()
        test_correct = 0
        test_total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = outputs.max(1)
                test_total += labels.size(0)
                test_correct += predicted.eq(labels).sum().item()
        
        test_acc = 100.0 * test_correct / test_total
        history['phase'].append(f'main_{epoch+1}')
        history['test_acc'].append(test_acc)
        print(f"  Main Epoch {epoch+1}/{main_epochs} - Test Acc: {test_acc:.2f}%")
    
    return history

# Train with transfer curriculum
set_seed(42)
transfer_model = SimpleCNN()

transfer_history = train_transfer_curriculum(
    model=transfer_model,
    auxiliary_dataset=gray_train_dataset,
    main_dataset=train_dataset,
    test_loader=test_loader,
    aux_epochs=5,
    main_epochs=10,
    batch_size=128,
    lr=0.001,
    device=device
)

### Visualize transfer curriculum progression

Notice the transition from auxiliary to main task.

In [ ]:
plt.figure(figsize=(10, 5))
x = range(len(transfer_history['test_acc']))
plt.plot(x, transfer_history['test_acc'], marker='o')

# Mark phase transition
aux_epochs = sum(1 for p in transfer_history['phase'] if p.startswith('aux'))
plt.axvline(x=aux_epochs - 0.5, color='red', linestyle='--', label='Phase Transition')

plt.xlabel('Training Step')
plt.ylabel('Test Accuracy (%)')
plt.title('Transfer Curriculum: Auxiliary → Main Task')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nFinal test accuracy: {transfer_history['test_acc'][-1]:.2f}%")

## 7. When Does Curriculum Learning Help?

Curriculum learning doesn't always help. Let's summarize **when it's beneficial**:

### 1. Noisy Data
- **Problem**: Noisy labels or corrupted examples can mislead early training
- **Solution**: Start with clean/confident examples, gradually introduce noisy ones
- **Example**: Medical imaging with uncertain diagnoses

### 2. Hard Optimization Landscapes
- **Problem**: Complex loss surfaces with many local minima
- **Solution**: Easy examples help find good initialization basins
- **Example**: Deep reinforcement learning, non-convex objectives

### 3. Domain Shift
- **Problem**: Target distribution differs from source
- **Solution**: Gradually transition from source to target domain
- **Example**: Synthetic → real data, simulation → real-world

### 4. Limited Data
- **Problem**: Few training examples
- **Solution**: Structured progression helps extract more from limited data
- **Example**: Few-shot learning, rare disease classification

### 5. Sequence Length
- **Problem**: Long sequences are hard to learn from scratch
- **Solution**: Start with short sequences, increase length
- **Example**: Machine translation, protein folding

### Demonstrating curriculum with noisy labels

Let's simulate a noisy dataset and show how curriculum helps.

In [ ]:
class NoisyCIFAR10(Dataset):
    """
    CIFAR-10 with random label noise.
    """
    def __init__(self, base_dataset, noise_ratio: float = 0.3, seed: int = 42):
        self.base_dataset = base_dataset
        self.noise_ratio = noise_ratio
        
        # Randomly corrupt labels
        np.random.seed(seed)
        n = len(base_dataset)
        self.is_noisy = np.random.rand(n) < noise_ratio
        
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        image, label = self.base_dataset[idx]
        
        # Corrupt label if marked as noisy
        if self.is_noisy[idx]:
            # Random label
            label = np.random.randint(0, 10)
        
        return image, label

# Create noisy dataset
noisy_train_dataset = NoisyCIFAR10(train_dataset, noise_ratio=0.3)
noise_count = noisy_train_dataset.is_noisy.sum()
print(f"Created noisy dataset with {noise_count}/{len(noisy_train_dataset)} corrupted labels")
print(f"Noise ratio: {100 * noise_count / len(noisy_train_dataset):.1f}%")

### Difficulty metric: predict clean vs noisy

We can use a small model to identify likely clean examples.

In [ ]:
def compute_noise_difficulty(
    dataset: Dataset,
    model: nn.Module,
    device: torch.device,
    batch_size: int = 256
) -> np.ndarray:
    """
    Use model loss to estimate which examples are likely noisy.
    Higher loss = likely noisy = harder.
    """
    model.eval()
    all_losses = []
    
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            losses = F.cross_entropy(outputs, labels, reduction='none')
            all_losses.append(losses.cpu().numpy())
    
    return np.concatenate(all_losses)

# Train a small model to estimate difficulties
set_seed(42)
probe_model = SimpleCNN()
print("Training probe model to identify noisy examples...")
_ = train_with_curriculum(
    model=probe_model,
    dataset=noisy_train_dataset,
    test_loader=test_loader,
    scheduler=None,
    epochs=3,  # Just a few epochs
    batch_size=128,
    lr=0.001,
    device=device
)

# Compute difficulties
noise_difficulties = compute_noise_difficulty(noisy_train_dataset, probe_model, device)
print(f"\nDifficulty scores computed!")

### Analyze difficulty vs noise

Let's verify that high-loss examples are indeed noisy.

In [ ]:
# Sort by difficulty
sorted_indices = np.argsort(noise_difficulties)

# Check noise ratio in easy vs hard examples
easy_indices = sorted_indices[:len(sorted_indices)//2]
hard_indices = sorted_indices[len(sorted_indices)//2:]

easy_noise_ratio = noisy_train_dataset.is_noisy[easy_indices].mean()
hard_noise_ratio = noisy_train_dataset.is_noisy[hard_indices].mean()

print(f"Noise ratio in 'easy' half (low loss): {100*easy_noise_ratio:.1f}%")
print(f"Noise ratio in 'hard' half (high loss): {100*hard_noise_ratio:.1f}%")
print(f"\nThe high-loss examples are {hard_noise_ratio/easy_noise_ratio:.1f}x more likely to be noisy!")

This confirms that our difficulty metric successfully identifies noisy examples — they have higher loss.

### Train with noise-aware curriculum

In [ ]:
# Curriculum that avoids noisy examples initially
noise_curriculum_config = CurriculumConfig(
    strategy='baby_steps',
    initial_ratio=0.4,  # Start with cleanest 40%
    growth_rate=0.1,
    warmup_epochs=3
)

noise_scheduler = CurriculumScheduler(noise_difficulties, noise_curriculum_config)

set_seed(42)
noise_curriculum_model = SimpleCNN()

print("Training with noise-aware curriculum...")
noise_curriculum_history = train_with_curriculum(
    model=noise_curriculum_model,
    dataset=noisy_train_dataset,
    test_loader=test_loader,
    scheduler=noise_scheduler,
    epochs=12,
    batch_size=128,
    lr=0.001,
    device=device
)

### Baseline on noisy data

In [ ]:
set_seed(42)
noise_baseline_model = SimpleCNN()

print("Training baseline on noisy data...")
noise_baseline_history = train_with_curriculum(
    model=noise_baseline_model,
    dataset=noisy_train_dataset,
    test_loader=test_loader,
    scheduler=None,
    epochs=12,
    batch_size=128,
    lr=0.001,
    device=device
)

### Compare performance on noisy data

In [ ]:
plt.figure(figsize=(10, 5))
epochs = range(1, len(noise_baseline_history['test_acc']) + 1)

plt.plot(epochs, noise_baseline_history['test_acc'], marker='o', label='Baseline (Noisy)')
plt.plot(epochs, noise_curriculum_history['test_acc'], marker='s', label='Curriculum (Noisy)')
plt.axhline(y=baseline_history['test_acc'][-1], color='gray', linestyle='--', 
            label='Clean Data Reference', alpha=0.7)

plt.xlabel('Epoch')
plt.ylabel('Test Accuracy (%)')
plt.title('Learning from Noisy Labels')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\nFinal Test Accuracy (Noisy Data):")
print(f"Baseline: {noise_baseline_history['test_acc'][-1]:.2f}%")
print(f"Curriculum: {noise_curriculum_history['test_acc'][-1]:.2f}%")
print(f"Improvement: {noise_curriculum_history['test_acc'][-1] - noise_baseline_history['test_acc'][-1]:.2f}%")

**Key Takeaway**: By starting with clean examples and gradually introducing noisy ones, curriculum learning helps the model build robust features before encountering corrupted labels.

## 8. Anti-Curriculum: Learning Hard Examples First

The **anti-curriculum** is a controversial idea: what if we train on hard examples first?

**Hypothesis**: Hard examples force the model to learn more discriminative features early on.

**Counter-hypothesis**: Hard examples might lead to bad local minima if the model isn't ready.

Let's test it!

### Anti-curriculum scheduler

In [ ]:
class AntiCurriculumScheduler:
    """
    Start with hardest examples, gradually add easier ones.
    """
    def __init__(self, difficulty_scores: np.ndarray, config: CurriculumConfig):
        self.difficulty_scores = difficulty_scores
        self.config = config
        # Sort indices by difficulty (HARD to EASY)
        self.sorted_indices = np.argsort(difficulty_scores)[::-1]  # Reverse order!
        self.n_examples = len(difficulty_scores)
    
    def get_indices(self, epoch: int) -> np.ndarray:
        if epoch < self.config.warmup_epochs:
            ratio = self.config.initial_ratio
        else:
            epochs_since_warmup = epoch - self.config.warmup_epochs
            ratio = min(
                self.config.final_ratio,
                self.config.initial_ratio + epochs_since_warmup * self.config.growth_rate
            )
        
        n_examples = int(self.n_examples * ratio)
        # Take hardest n_examples
        indices = self.sorted_indices[:n_examples]
        np.random.shuffle(indices)
        return indices

# Create anti-curriculum
anti_config = CurriculumConfig(
    strategy='baby_steps',
    initial_ratio=0.3,
    growth_rate=0.1,
    warmup_epochs=3
)

anti_scheduler = AntiCurriculumScheduler(train_difficulties, anti_config)

# Verify it starts with hard examples
first_epoch_indices = anti_scheduler.get_indices(0)
first_epoch_difficulties = train_difficulties[first_epoch_indices]
print(f"Anti-curriculum first epoch difficulty range: [{first_epoch_difficulties.min():.4f}, {first_epoch_difficulties.max():.4f}]")
print(f"Overall difficulty range: [{train_difficulties.min():.4f}, {train_difficulties.max():.4f}]")
print(f"Starting with hardest examples: {first_epoch_difficulties.mean() > train_difficulties.mean()}")

### Train with anti-curriculum

In [ ]:
set_seed(42)
anti_model = SimpleCNN()

print("Training with anti-curriculum (hard to easy)...")
anti_history = train_with_curriculum(
    model=anti_model,
    dataset=train_dataset,
    test_loader=test_loader,
    scheduler=anti_scheduler,
    epochs=15,
    batch_size=128,
    lr=0.001,
    device=device
)

### Compare curriculum vs anti-curriculum

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs = range(1, len(baseline_history['test_acc']) + 1)

# Test accuracy
axes[0].plot(epochs, baseline_history['test_acc'], marker='o', label='Baseline (Random)', alpha=0.7)
axes[0].plot(epochs, curriculum_history['test_acc'], marker='s', label='Curriculum (Easy→Hard)')
axes[0].plot(epochs, anti_history['test_acc'], marker='^', label='Anti-Curriculum (Hard→Easy)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title('Curriculum vs Anti-Curriculum')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Training loss
axes[1].plot(epochs, baseline_history['train_loss'], marker='o', label='Baseline (Random)', alpha=0.7)
axes[1].plot(epochs, curriculum_history['train_loss'], marker='s', label='Curriculum (Easy→Hard)')
axes[1].plot(epochs, anti_history['train_loss'], marker='^', label='Anti-Curriculum (Hard→Easy)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Training Loss')
axes[1].set_title('Training Loss Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nFinal Results:")
print(f"Baseline: {baseline_history['test_acc'][-1]:.2f}%")
print(f"Curriculum (Easy→Hard): {curriculum_history['test_acc'][-1]:.2f}%")
print(f"Anti-Curriculum (Hard→Easy): {anti_history['test_acc'][-1]:.2f}%")

**Observation**: Anti-curriculum often performs worse than curriculum learning. Starting with hard examples can:
- Lead to slower initial learning (higher early loss)
- Potentially get stuck in bad local minima
- Miss the benefits of building strong fundamentals first

However, there are rare cases where anti-curriculum can help (e.g., when easy examples are misleading), but generally **easy-to-hard** is more reliable.

## 9. Key Takeaways

### Core Concepts

1. **Curriculum learning mimics human education** — start simple, build complexity gradually
2. **Difficulty can be measured many ways**:
   - Domain knowledge (length, variance, complexity)
   - Model-based (loss, confidence)
   - Ensemble disagreement
3. **Multiple curriculum strategies exist**:
   - Fixed schedule (baby steps, spiral)
   - Self-paced (model chooses)
   - Transfer (auxiliary → main task)

### When It Helps

- **Noisy data**: Learn from clean examples first
- **Hard optimization**: Find better basins early
- **Domain shift**: Gradual adaptation
- **Sequence tasks**: Start with short sequences
- **Limited data**: Extract more signal

### Practical Insights

- **Easy-to-hard generally outperforms hard-to-easy**
- **The difficulty metric matters** — choose based on your domain
- **Growth rate is a key hyperparameter** — too fast loses benefits, too slow wastes time
- **Self-paced learning adapts automatically** but requires a good initial model
- **Transfer curricula work well for domain shift** problems

### What We Built

✓ Multiple difficulty scoring methods  
✓ Curriculum scheduler with configurable strategies  
✓ Self-paced learning algorithm  
✓ Image classification experiments (CIFAR-10)  
✓ Sequence learning experiments (arithmetic)  
✓ Transfer curriculum (grayscale → color)  
✓ Noisy label robustness demonstration  
✓ Anti-curriculum comparison  

Curriculum learning is a powerful training paradigm that can significantly improve model performance when applied thoughtfully!